In [0]:
# Core
from pyspark.sql import SparkSession

# Functions
from pyspark.sql.functions import (
    col, sum, expr, when, count,
    radians, sin, cos, sqrt, atan2, when
)
from pyspark.sql import Row

# Visualization (convert ke pandas)
import matplotlib.pyplot as plt
import pandas as pd

In [0]:
df = spark.read.csv(
    "/Volumes/train/default/train_data/",
    header=True,
    inferSchema=True,
    multiLine=True
)
df.show(5)

In [0]:
print("Jumlah data:", df.count())
df.printSchema()
df.explain()

In [0]:
df.describe().show()

In [0]:
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [0]:
df.groupBy("passenger_count").count().orderBy("passenger_count").show()

In [0]:
df.groupBy("fare_amount").count().orderBy("fare_amount").show()

In [0]:
print("Fare negatif:", df.filter(col("fare_amount") < 0).count())
print("Fare terlalu tinggi:", df.filter(col("fare_amount") > 500).count())

print("Passenger 0:", df.filter(col("passenger_count") == 0).count())
print("Passenger > 6:", df.filter(col("passenger_count") > 6).count())

In [0]:
df.filter(
    (col("pickup_latitude") == 0) | (col("pickup_longitude") == 0)
).count()

In [0]:
df_clean = df
df_clean = df_clean.withColumn( 
    "passenger_count", when((col("passenger_count") <= 0) | (col("passenger_count") > 6), None) 
    .otherwise(col("passenger_count")) 
) 
df_clean = df_clean.withColumn( 
    "fare_amount", when((col("fare_amount") <= 0) | (col("fare_amount") > 500), None) 
    .otherwise(col("fare_amount")) 
)

In [0]:
df_clean = df_clean.dropna(subset=["fare_amount"])

In [0]:
fill_values = {}

for field in df_clean.schema.fields:
    if field.name != "fare_amount":
        if field.dataType.simpleString() in ['int', 'double', 'float']:
            median = df_clean.approxQuantile(field.name, [0.5], 0.01)[0]
            fill_values[field.name] = median

df_clean = df_clean.fillna(fill_values)

In [0]:
df_clean = df_clean.filter(
    (col("pickup_latitude").between(40.6, 40.9)) &
    (col("dropoff_latitude").between(40.6, 40.9)) &
    (col("pickup_longitude").between(-74.1, -73.7)) &
    (col("dropoff_longitude").between(-74.1, -73.7))
)

In [0]:
R = 6371

df_clean = df_clean.withColumn(
    "distance_km",
    R * 2 * atan2(
        sqrt(
            sin((radians(col("dropoff_latitude")) - radians(col("pickup_latitude"))) / 2) ** 2 +
            cos(radians(col("pickup_latitude"))) *
            cos(radians(col("dropoff_latitude"))) *
            sin((radians(col("dropoff_longitude")) - radians(col("pickup_longitude"))) / 2) ** 2
        ),
        sqrt(1 - (
            sin((radians(col("dropoff_latitude")) - radians(col("pickup_latitude"))) / 2) ** 2 +
            cos(radians(col("pickup_latitude"))) *
            cos(radians(col("dropoff_latitude"))) *
            sin((radians(col("dropoff_longitude")) - radians(col("pickup_longitude"))) / 2) ** 2
        ))
    )
)

In [0]:
print("Distance 0:", df_clean.filter(col("distance_km") == 0).count())

In [0]:
df_clean = df_clean.filter(col("distance_km") > 0)

In [0]:
df_clean.count()

In [0]:
df_clean.select(
    sum((col("fare_amount") <= 0).cast("int")).alias("fare_anomali"),

    sum(((col("passenger_count") <= 0) | (col("passenger_count") > 6)).cast("int"))
        .alias("passenger_anomali"),

    sum(((col("distance_km") == 0) | (col("distance_km") > 100)).cast("int"))
        .alias("distance_anomali"),

    sum(((col("pickup_longitude") < -74.1) | (col("pickup_longitude") > -73.7)).cast("int"))
        .alias("pickup_longitude_anomali"),

    sum(((col("dropoff_longitude") < -74.1) | (col("dropoff_longitude") > -73.7)).cast("int"))
        .alias("dropoff_longitude_anomali"),

    sum(((col("pickup_latitude") < 40.6) | (col("pickup_latitude") > 40.9)).cast("int"))
        .alias("pickup_latitude_anomali"),

    sum(((col("dropoff_latitude") < 40.6) | (col("dropoff_latitude") > 40.9)).cast("int"))
        .alias("dropoff_latitude_anomali")

).show()

In [0]:
pdf = df_clean.select(
    "fare_amount", "distance_km", "passenger_count"
).limit(10000).toPandas()

In [0]:
def plot_heatmap_spark(df, sample_frac=0.1, max_rows=10000):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from pyspark.sql.types import NumericType

    numeric_cols = [
        field.name for field in df.schema.fields
        if isinstance(field.dataType, NumericType)
    ]

    pdf = (
        df.select(numeric_cols)
        .sample(sample_frac)
        .limit(max_rows)
        .toPandas()
    )

    corr = pdf.corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        linewidths=0.5,
        linecolor="white"
    )

    plt.title("Correlation Heatmap", fontsize=14)
    plt.tight_layout()
    plt.show()

In [0]:
plot_heatmap_spark(df_clean)

In [0]:
df_clean.describe().show()
print("Jumlah data:", df_clean.count())